# Exercises: Trajectory Requests

These exercises build on `polytope/trajectory.ipynb`. The focus is on writing **correct Polytope requests** (not on plotting or earthkit-specific syntax), using operational forecast data (`class: od`, `stream: oper`/`enfo`) and AIFS Single (`class: ai`, `stream: oper`).

Useful reference: [Trajectory feature docs](https://polytope.readthedocs.io/en/latest/Service/Features/trajectory/)

Each exercise gives you a request skeleton with `# TODO` markers, or a broken request for you to debug. A cell is provided after each request to actually submit it to the Polytope server — run it to check your request works, but the main goal is still getting the request dictionary right.

In [ ]:
import earthkit.data

### 1) Build a basic trajectory request

Request **maximum 10m wind gust (049)** as a trajectory along a ferry route crossing the English Channel, with waypoints at Dover (51.13N, 1.31E), mid-Channel (50.85N, 1.20E) and Calais (50.97N, 1.85E). Use the deterministic forecast, with an inflation of 0.2° circular radius  around the route.

In [ ]:
request_1 = {
    "class": "od",
    "stream": "oper",
    "type": "fc",
    "date": "-1",
    "time": "0000",
    "expver": "0001",
    "domain": "g",
    "levtype": "sfc",
    "param": "049",
    "step": "0",
    # TODO: add a "feature" dictionary of type "trajectory" with the 3 waypoints above,

    # "axes": ["latitude", "longitude"] and inflation 0.2
    }

In [ ]:
data_1 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_1, address="polytope.ecmwf.int", stream=False)
data_1.to_xarray()

### 2) This request raises an error. Find out why and fix it.

The forecaster wants wind speed along a short hiking trail near Zermatt.

In [ ]:
request_2 = {
    "class": "od",
    "stream": "oper",
    "type": "fc",
    "date": "-1",
    "time": "0000",
    "expver": "0001",
    "domain": "g",
    "levtype": "sfc",
    "param": "165/166",
    "step": "0",
    "feature": {
        "type": "trajectory",
        "points": [[46.02, 7.75]],
        "axes": ["latitude", "longitude"],
        "inflation": 0.1,

    },}

In [ ]:
data_2 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_2, address="polytope.ecmwf.int", stream=False)
data_2.to_xarray()

### 3) Non-trivial: anisotropic box inflation along a narrow valley

The Inn valley near Innsbruck is narrow, so a circular buffer around a route through it would spill over onto the surrounding mountains. Build a trajectory for **2m temperature (167)** between Innsbruck (47.26N, 11.40E) and Kufstein (47.58N, 12.17E) from the deterministic forecast, using a `box` inflation that is narrow across the valley but wider along it: 0.05° in latitude, 0.3° in longitude.

In [ ]:
request_3 = {
    "class": "od",
    "stream": "oper",
    "type": "fc",
    "date": "-1",
    "time": "0000",
    "expver": "0001",
    "domain": "g",
    "levtype": "sfc",
    "param": "167",
    "step": "0",
    "feature": {
        "type": "trajectory",
        "points": [[47.26, 11.40], [47.58, 12.17]],
        "axes": ["latitude", "longitude"],
        # TODO: add "inflate": "box" and "inflation" as a list of 2 values (lat, lon)
    },
}

In [ ]:
data_3 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_3, address="polytope.ecmwf.int", stream=False)
data_3.to_xarray()

### 4) This request raises an error. Find out why and fix it.

This is meant to track 2m temperature along a time-evolving cold front over 48 hours of a single AIFS Single forecast, inflating a box of ±0.3° latitude, ±0.3° longitude and ±6h of step around each point.

In [ ]:
request_4 = {
    "class": "ai",
    "model": "aifs-single",
    "date": "-3",
    "expver": 1,
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "levtype": "sfc",
    "param": "167",
    "feature": {
        "type": "trajectory",
        "points": [[54.0, 9.0, 0], [52.0, 10.0, 24], [50.0, 11.0, 48]],
        "axes": ["latitude", "longitude", "step"],
        "inflate": "box",
        "inflation": 0.3,
    },
}

In [ ]:
data_4 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_4, address="polytope.ecmwf.int", stream=False)
data_4.to_xarray()

### 5) Non-trivial: a storm-track trajectory with ensemble members

A low-pressure system is forecast to track from (54.0N, -8.0E) to (58.0N, -3.0E) to (60.0N, 10.0E) over the next three days (Ireland → Scotland → Norway). Build a trajectory request for **mean sea level pressure (151)** along this track for ensemble members 1 to 20, using an inflation of 1.0° radius to capture uncertainty in the track position.

In [ ]:
request_5 = {
    "class": "od",
    "stream": "enfo",
    "type": "pf",
    "date": "-1",
    "time": "0000",
    "expver": "0001",
    "domain": "g",
    "levtype": "sfc",
    "param": "151",
    "step": "0",
    "number": "1/to/20",
    # TODO: add a "feature" dictionary of type "trajectory" with the 3 track points,
    # "axes": ["latitude", "longitude"] and inflation 1.0
}

In [ ]:
data_5 = earthkit.data.from_source(
    "polytope", "ecmwf-mars", request_5, address="polytope.ecmwf.int", stream=False)
data_5.to_xarray()